In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv("../data/raw/housing.csv")

In [ ]:
X=df.drop('median_house_value',axis=1)
y=df['median_house_value']

In [ ]:
print(X.shape)
print(y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
print(X_train.shape)
print(X_test.shape)

In [ ]:
X_train.isnull().sum()

In [ ]:
numerical_columns = X_train.select_dtypes(include=["number"]).columns
categorical_columns = X_train.select_dtypes(include=["object"]).columns

In [ ]:
print("Numerical Columns:")
print(numerical_columns)

print("\nCategorical Columns:")
print(categorical_columns)

In [ ]:
X_train['ocean_proximity'].unique()

In [ ]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="median")
X_train["total_bedrooms"] = imputer.fit_transform(
    X_train[["total_bedrooms"]]
)
X_test["total_bedrooms"] = imputer.transform(
    X_test[["total_bedrooms"]]
)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False,handle_unknown="ignore")

In [ ]:
encoder.fit(X_train[["ocean_proximity"]])
encoded_train=encoder.transform(X_train[["ocean_proximity"]])
encoded_test=encoder.transform(X_test[["ocean_proximity"]])

In [ ]:
print(encoded_train.shape)
print(encoded_train)

In [ ]:
encoded_column_names = encoder.get_feature_names_out(["ocean_proximity"])
print(encoded_column_names)

In [ ]:
encoded_train_df = pd.DataFrame(
    encoded_train,
    columns=encoded_column_names,
    index=X_train.index
)

encoded_test_df = pd.DataFrame(
    encoded_test,
    columns=encoded_column_names,
    index=X_test.index
)

In [ ]:
X_train = X_train.drop("ocean_proximity", axis=1)
X_test = X_test.drop("ocean_proximity", axis=1)

In [ ]:
X_train = pd.concat([X_train, encoded_train_df], axis=1)
X_test = pd.concat([X_test, encoded_test_df], axis=1)

In [ ]:
print(X_train.head())
print(X_train.shape)
print(X_test.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
scaler.fit(X_train)
X_train_scaled=scaler.transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train_scaled, y_train)

In [ ]:
y_pred=model.predict(X_test_scaled)

In [ ]:
from sklearn.metrics import (mean_absolute_error,root_mean_squared_error,mean_squared_error,r2_score)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R²  :", r2)

In [ ]:
print("Intercept:", model.intercept_)
coefficients = pd.DataFrame({"Feature": X_train.columns,"Coefficient": model.coef_})
print(coefficients.sort_values(by="Coefficient", ascending=False))

In [ ]:
plt.figure(figsize=(10, 7))
plt.scatter(
    y_test,
    y_pred,
    alpha=0.5,
    edgecolor="k",
    linewidth=0.2
)

min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    color="red",
    linestyle="--",
    linewidth=2,
    label="Perfect Prediction"
)

plt.title("Linear Regression: Actual vs Predicted House Prices", fontsize=15)
plt.xlabel("Actual House Price")
plt.ylabel("Predicted House Price")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
plt.savefig(
    r"D:\AI-ML Code & Projects\Projects\House Price Prediction\reports\figures\actual_vs_predicted.png",
    dpi=300,
    bbox_inches="tight"
)

In [ ]:
residuals = y_test - y_pred
plt.figure(figsize=(10, 7))
plt.scatter(
    y_pred,
    residuals,
    alpha=0.5,
    edgecolor="k",
    linewidth=0.2
)

plt.axhline(
    y=0,
    color="red",
    linestyle="--",
    linewidth=2
)

plt.title("Linear Regression Residual Analysis", fontsize=15)
plt.xlabel("Predicted House Price")
plt.ylabel("Residual")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
plt.savefig(
    r"D:\AI-ML Code & Projects\Projects\House Price Prediction\reports\figures\residual_plot.png",
    dpi=300,
    bbox_inches="tight"
)

In [ ]:
coef_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": model.coef_
})
coef_df = coef_df.sort_values("Coefficient")

plt.figure(figsize=(11, 7))
plt.barh(
    coef_df["Feature"],
    coef_df["Coefficient"]
)
plt.title("Feature Importance (Linear Regression Coefficients)", fontsize=15)
plt.xlabel("Coefficient")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()
plt.savefig(
    r"D:\AI-ML Code & Projects\Projects\House Price Prediction\reports\figures/coefficient_plot.png",
    dpi=300,
    bbox_inches="tight"
)

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(
    df.corr(numeric_only=True),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    linewidths=0.5
)
plt.title("Correlation Heatmap", fontsize=15)
plt.tight_layout()
plt.show()
plt.savefig(
    r"D:\AI-ML Code & Projects\Projects\House Price Prediction\reports\figures/correlation_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)